<a href="https://colab.research.google.com/github/jwork3838-cloud/MLB-hr-engine/blob/main/HRpropModel_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# MLB Home Run Predictor – Daily Prediction (v2, 34 features)
# Uses expanded training data from Google Sheet.
# ============================================================================

import pandas as pd
import numpy as np
import requests
import time
import re
import urllib.parse
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import gspread
from google.colab import auth, drive
from google.auth import default
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------------
TODAY = datetime.now().strftime('%Y-%m-%d')
OPENWEATHER_API_KEY = "3b1c666e88254b0827f0f37e326aa46f"
SHEET_ID = "10ULW_blNg-yPEdruSUcoTe1fhyOwr_fNnoZGF7PWRw0"   # v2 sheet
ROLLING_DAYS = None   # use all training data (set to e.g., 60 for recent only)

# ----------------------------------------------------------------------------
# HEADERS FOR SAVANT REQUESTS
# ----------------------------------------------------------------------------
SAVANT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
}

# ----------------------------------------------------------------------------
# HARDCODED DATA (park factors, stadiums) – all 30 teams
# ----------------------------------------------------------------------------
parkFactorHR = {
    "ARI": 1.12, "ATL": 1.05, "BAL": 0.98, "BOS": 1.06, "CHC": 0.95,
    "CWS": 0.92, "CIN": 1.23, "CLE": 0.88, "COL": 1.42, "DET": 1.00,
    "HOU": 1.09, "KC": 1.02, "LAA": 1.01, "LAD": 1.08, "MIA": 0.91,
    "MIL": 1.10, "MIN": 1.04, "NYM": 0.96, "NYY": 1.13, "ATH": 0.97,
    "PHI": 1.07, "PIT": 0.94, "SD": 0.89, "SF": 0.85, "SEA": 0.93,
    "STL": 0.99, "TB": 0.90, "TEX": 1.11, "TOR": 1.08, "WSH": 1.02
}

stadiumData = {
    "Diamondbacks": {"lat": 33.445, "lon": -112.067, "isDome": True, "abbrev": "ARI", "teamId": 109},
    "Braves":       {"lat": 33.891, "lon": -84.468,  "isDome": False, "abbrev": "ATL", "teamId": 144},
    "Orioles":      {"lat": 39.284, "lon": -76.622,  "isDome": False, "abbrev": "BAL", "teamId": 110},
    "Red Sox":      {"lat": 42.346, "lon": -71.097,  "isDome": False, "abbrev": "BOS", "teamId": 111},
    "Cubs":         {"lat": 41.948, "lon": -87.656,  "isDome": False, "abbrev": "CHC", "teamId": 112},
    "White Sox":    {"lat": 41.830, "lon": -87.634,  "isDome": False, "abbrev": "CWS", "teamId": 145},
    "Reds":         {"lat": 39.097, "lon": -84.507,  "isDome": False, "abbrev": "CIN", "teamId": 113},
    "Guardians":    {"lat": 41.496, "lon": -81.685,  "isDome": False, "abbrev": "CLE", "teamId": 114},
    "Rockies":      {"lat": 39.756, "lon": -104.994, "isDome": False, "abbrev": "COL", "teamId": 115},
    "Tigers":       {"lat": 42.339, "lon": -83.049,  "isDome": False, "abbrev": "DET", "teamId": 116},
    "Astros":       {"lat": 29.757, "lon": -95.356,  "isDome": True,  "abbrev": "HOU", "teamId": 117},
    "Royals":       {"lat": 39.051, "lon": -94.480,  "isDome": False, "abbrev": "KC",  "teamId": 118},
    "Angels":       {"lat": 33.800, "lon": -117.883, "isDome": False, "abbrev": "LAA", "teamId": 108},
    "Dodgers":      {"lat": 34.074, "lon": -118.240, "isDome": False, "abbrev": "LAD", "teamId": 119},
    "Marlins":      {"lat": 25.778, "lon": -80.220,  "isDome": True,  "abbrev": "MIA", "teamId": 146},
    "Brewers":      {"lat": 43.028, "lon": -87.971,  "isDome": True,  "abbrev": "MIL", "teamId": 158},
    "Twins":        {"lat": 44.982, "lon": -93.278,  "isDome": False, "abbrev": "MIN", "teamId": 142},
    "Mets":         {"lat": 40.757, "lon": -73.846,  "isDome": False, "abbrev": "NYM", "teamId": 121},
    "Yankees":      {"lat": 40.829, "lon": -73.926,  "isDome": False, "abbrev": "NYY", "teamId": 147},
    "Athletics":    {"lat": 38.580, "lon": -121.514, "isDome": False, "abbrev": "ATH", "teamId": 133},
    "Phillies":     {"lat": 39.906, "lon": -75.167,  "isDome": False, "abbrev": "PHI", "teamId": 143},
    "Pirates":      {"lat": 40.447, "lon": -80.006,  "isDome": False, "abbrev": "PIT", "teamId": 134},
    "Padres":       {"lat": 32.707, "lon": -117.157, "isDome": False, "abbrev": "SD",  "teamId": 135},
    "Giants":       {"lat": 37.779, "lon": -122.389, "isDome": False, "abbrev": "SF",  "teamId": 137},
    "Mariners":     {"lat": 47.591, "lon": -122.333, "isDome": True,  "abbrev": "SEA", "teamId": 136},
    "Cardinals":    {"lat": 38.623, "lon": -90.193,  "isDome": False, "abbrev": "STL", "teamId": 138},
    "Rays":         {"lat": 27.768, "lon": -82.653,  "isDome": True,  "abbrev": "TB",  "teamId": 139},
    "Rangers":      {"lat": 32.751, "lon": -97.083,  "isDome": True,  "abbrev": "TEX", "teamId": 140},
    "Blue Jays":    {"lat": 43.641, "lon": -79.390,  "isDome": True,  "abbrev": "TOR", "teamId": 141},
    "Nationals":    {"lat": 38.873, "lon": -77.007,  "isDome": False, "abbrev": "WSH", "teamId": 120},
}

# ----------------------------------------------------------------------------
# HELPER FUNCTIONS (fetch, API, CSV)
# ----------------------------------------------------------------------------
def fetch_json(url):
    try:
        r = requests.get(url, headers=SAVANT_HEADERS, timeout=15)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"❌ fetchJSON error: {e}")
        return None

def fetch_csv(url):
    try:
        r = requests.get(url, headers=SAVANT_HEADERS, timeout=30)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f"❌ fetchCSV error: {e}")
        return None

def get_pitcher_hand(pid):
    data = fetch_json(f"https://statsapi.mlb.com/api/v1/people/{pid}")
    if data and 'people' in data:
        return data['people'][0].get('pitchHand', {}).get('code')
    return None

def get_batter_hand(bid):
    data = fetch_json(f"https://statsapi.mlb.com/api/v1/people/{bid}")
    if data and 'people' in data:
        return data['people'][0].get('batSide', {}).get('code')
    return None

def calculate_woba_from_stat(stat):
    try:
        ab = stat.get('atBats', 0)
        bb = stat.get('baseOnBalls', 0)
        hbp = stat.get('hitByPitch', 0)
        singles = stat.get('singles', 0)
        doubles = stat.get('doubles', 0)
        triples = stat.get('triples', 0)
        hr = stat.get('homeRuns', 0)
        woba = (0.69*bb + 0.69*hbp + 0.89*singles + 1.27*doubles + 1.62*triples + 2.10*hr) / (ab+bb+hbp) if (ab+bb+hbp) else 0
        return woba
    except:
        return 0

def calculate_iso_from_stat(stat):
    try:
        slg = stat.get('slugging', 0)
        avg = stat.get('avg', 0)
        return slg - avg if slg and avg else 0
    except:
        return 0

def get_batter_platoon_splits(bid):
    try:
        url = f"https://statsapi.mlb.com/api/v1/people/{bid}/stats?stats=statSplits&sitCodes=vl,vr&season=2026&group=hitting"
        data = fetch_json(url)
        if not data or 'stats' not in data:
            return {}
        splits = data['stats'][0].get('splits', [])
        result = {}
        for split in splits:
            code = split.get('split', {}).get('code')
            stat = split.get('stat', {})
            if code == 'vl':
                result['wOBA_vs_L'] = calculate_woba_from_stat(stat)
                result['ISO_vs_L'] = calculate_iso_from_stat(stat)
            elif code == 'vr':
                result['wOBA_vs_R'] = calculate_woba_from_stat(stat)
                result['ISO_vs_R'] = calculate_iso_from_stat(stat)
        return result
    except Exception:
        return {}

def get_roster_batters(tid):
    data = fetch_json(f"https://statsapi.mlb.com/api/v1/teams/{tid}/roster?season=2026")
    if not data or 'roster' not in data:
        return []
    return [p['person']['id'] for p in data['roster'] if p['position']['type'] != 'Pitcher']

def get_hot_streak(bid, target_date, days=7):
    try:
        start = (datetime.strptime(target_date, '%Y-%m-%d') - timedelta(days=days)).strftime('%Y-%m-%d')
        url = f"https://statsapi.mlb.com/api/v1/people/{bid}/stats?stats=gameLog&group=hitting&startDate={start}&endDate={target_date}"
        data = fetch_json(url)
        if not data or not data.get('stats') or not data['stats'][0].get('splits'):
            return None
        totals = {'pa':0, 'ab':0, 'singles':0, 'doubles':0, 'triples':0, 'hr':0, 'bb':0, 'hbp':0}
        for split in data['stats'][0]['splits']:
            s = split['stat']
            totals['pa'] += s.get('plateAppearances',0)
            totals['ab'] += s.get('atBats',0)
            totals['hr'] += s.get('homeRuns',0)
            totals['bb'] += s.get('baseOnBalls',0)
            totals['hbp'] += s.get('hitByPitch',0)
            hits = s.get('hits',0)
            doubles = s.get('doubles',0)
            triples = s.get('triples',0)
            totals['singles'] += hits - doubles - triples - totals['hr']
            totals['doubles'] += doubles
            totals['triples'] += triples
        if totals['pa'] == 0:
            return None
        iso = (totals['doubles'] + totals['triples']*2 + totals['hr']*3) / max(1, totals['ab'])
        woba = (0.69*(totals['bb']+totals['hbp']) + 0.89*totals['singles'] + 1.27*totals['doubles'] + 1.62*totals['triples'] + 2.10*totals['hr']) / totals['pa']
        hr_rate = totals['hr'] / totals['pa']
        return {'hot_pa': totals['pa'], 'hot_iso': iso, 'hot_woba': woba, 'hot_hr_rate': hr_rate}
    except Exception:
        return None

def parse_csv(csv_text):
    return pd.read_csv(pd.io.common.StringIO(csv_text)).to_dict(orient='records')

def safe_get(obj, key, default=0):
    if obj is None: return default
    return obj.get(key, default)

# ----------------------------------------------------------------------------
# FETCH PLATE DISCIPLINE (AUTOMATED)
# ----------------------------------------------------------------------------
def fetch_plate_discipline():
    url = "https://baseballsavant.mlb.com/leaderboard/custom?year=2026&type=batter&selections=player_id,last_name,first_name,z_swing_percent,oz_swing_percent,whiff_percent&min_pa=50&csv=true"
    headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"}
    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(pd.io.common.StringIO(r.text))
        df = df.rename(columns={
            "z_swing_percent": "z_swing_pct",
            "oz_swing_percent": "o_swing_pct",
            "whiff_percent": "whiff_pct"
        })
        df["contact_pct"] = 100 - df["whiff_pct"]
        plate_dict = {}
        for _, row in df.iterrows():
            pid = int(row["player_id"])
            plate_dict[pid] = {
                "o_swing_pct": row["o_swing_pct"],
                "z_swing_pct": row["z_swing_pct"],
                "contact_pct": row["contact_pct"]
            }
        print(f"✅ Fetched plate discipline for {len(plate_dict)} batters.")
        return plate_dict
    except Exception as e:
        print(f"⚠️ Could not fetch plate discipline: {e}")
        return {}

# ----------------------------------------------------------------------------
# LOAD STATIC STATCAST DATA (ONCE)
# ----------------------------------------------------------------------------
def load_static_csvs():
    print("Loading static Statcast CSVs...")
    batter_csv = fetch_csv("https://baseballsavant.mlb.com/statcast_leaderboard?type=batter&year=2026&position=&team=&min=q&csv=true")
    pitcher_csv = fetch_csv("https://baseballsavant.mlb.com/statcast_leaderboard?type=pitcher&year=2026&position=&team=&min=q&csv=true")
    expected_csv = fetch_csv("https://baseballsavant.mlb.com/leaderboard/expected_statistics?type=batter&year=2026&position=&team=&min=q&csv=true")
    batspeed_csv = fetch_csv("https://baseballsavant.mlb.com/leaderboard/bat-tracking?year=2026&csv=true")
    pitcher_exp_csv = fetch_csv("https://baseballsavant.mlb.com/leaderboard/expected_statistics?type=pitcher&year=2026&position=&team=&min=q&csv=true")
    pitch_movement_csv = fetch_csv("https://baseballsavant.mlb.com/leaderboard/pitch-movement?year=2026&csv=true")
    if not batter_csv or not pitcher_csv:
        raise ValueError("Failed to download core CSVs.")
    all_batters = {int(r['player_id']): r for r in parse_csv(batter_csv)}
    all_pitchers = {int(r['player_id']): r for r in parse_csv(pitcher_csv)}
    expected = {int(r['player_id']): r for r in parse_csv(expected_csv)}
    batspeed = {int(r['id']): r for r in parse_csv(batspeed_csv) if r.get('id')}
    pitcher_exp = {int(r['player_id']): r for r in parse_csv(pitcher_exp_csv)}
    # Pitch movement: aggregate per pitcher (fastball only)
    pitch_movement = parse_csv(pitch_movement_csv) if pitch_movement_csv else []
    pitcher_arsenal = {}
    for row in pitch_movement:
        pid = int(row['pitcher_id'])
        pitch_type = row.get('pitch_type', '').upper()
        if pitch_type in ('FF', '4-Seam Fastball'):
            pitcher_arsenal[pid] = {
                'fb_velo': row.get('avg_speed', 0),
                'spin_rate': row.get('spin_rate', 0),
                'horiz_break': row.get('horizontal_break', 0),
                'vert_break': row.get('vertical_break', 0)
            }
    print(f"Loaded: batters={len(all_batters)}, pitchers={len(all_pitchers)}, expected={len(expected)}, batspeed={len(batspeed)}, pitcher_exp={len(pitcher_exp)}, arsenal={len(pitcher_arsenal)}")
    return all_batters, all_pitchers, expected, batspeed, pitcher_exp, pitcher_arsenal

# ----------------------------------------------------------------------------
# FETCH TODAY'S PREDICTION FEATURES (NO TRAINING)
# ----------------------------------------------------------------------------
def fetch_todays_features(plate_dict, static_data):
    (all_batters, all_pitchers, expected, batspeed, pitcher_exp, pitcher_arsenal) = static_data
    print(f"\n📡 Fetching today's schedule ({TODAY}) and building features...")
    schedule = fetch_json(f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={TODAY}&hydrate=probablePitcher")
    if not schedule or not schedule.get('dates'):
        raise ValueError("No games found for today.")
    games = schedule['dates'][0].get('games', [])
    print(f"Found {len(games)} games.")

    starters = []
    team_ids = set()
    game_times = {}
    for g in games:
        away = g['teams']['away']['team']
        home = g['teams']['home']['team']
        team_ids.add(away['id']); team_ids.add(home['id'])
        game_time = g.get('gameDate')
        if game_time:
            hour = int(game_time[11:13])
            is_day = hour < 18
            game_times[g['gamePk']] = {'is_day': is_day}
        for side in ('away','home'):
            p = g['teams'][side].get('probablePitcher')
            if p:
                pitcher_name = p.get('fullName', 'Unknown')
                starters.append({
                    'id': p['id'],
                    'name': pitcher_name,
                    'teamId': g['teams'][side]['team']['id'],
                    'hand': None,
                    'gamePk': g['gamePk']
                })
    print(f"Probable starters: {len(starters)}")

    # Pitcher hands
    with ThreadPoolExecutor(max_workers=10) as ex:
        fut = {ex.submit(get_pitcher_hand, s['id']): s for s in starters}
        for f in as_completed(fut):
            s = fut[f]
            s['hand'] = f.result()
            print(f"{s['name']}: {s['hand'] or 'unknown'}")

    # Rosters
    print("\nFetching active rosters...")
    roster_batters = {}
    with ThreadPoolExecutor(max_workers=10) as ex:
        fut = {ex.submit(get_roster_batters, tid): tid for tid in team_ids}
        for f in as_completed(fut):
            tid = fut[f]
            batters = f.result()
            if batters:
                roster_batters[tid] = set(batters)
                print(f"Team {tid}: {len(batters)} batters")
            else:
                print(f"⚠️ No batters for team {tid}")

    # Batter hands, platoon splits & hot streaks
    all_batter_ids = set()
    for s in roster_batters.values():
        all_batter_ids.update(s)
    print(f"\nFetching batter handedness, platoon splits, hot streaks for {len(all_batter_ids)} batters...")
    batter_hand = {}
    batter_platoon = {}
    hot_streak = {}
    with ThreadPoolExecutor(max_workers=10) as ex:
        hand_fut = {ex.submit(get_batter_hand, bid): bid for bid in all_batter_ids}
        split_fut = {ex.submit(get_batter_platoon_splits, bid): bid for bid in all_batter_ids}
        streak_fut = {ex.submit(get_hot_streak, bid, TODAY): bid for bid in all_batter_ids}
        for f in as_completed(hand_fut):
            bid = hand_fut[f]
            try:
                batter_hand[bid] = f.result()
            except:
                batter_hand[bid] = None
        for f in as_completed(split_fut):
            bid = split_fut[f]
            try:
                batter_platoon[bid] = f.result()
            except:
                batter_platoon[bid] = {}
        for f in as_completed(streak_fut):
            bid = streak_fut[f]
            try:
                hot_streak[bid] = f.result()
            except:
                hot_streak[bid] = None
    print(f"Batter hands: {len(batter_hand)}, platoon splits: {len(batter_platoon)}, hot streaks: {len(hot_streak)}")

    print("\nBuilding feature rows...")
    features = []
    for g in games:
        away_id = g['teams']['away']['team']['id']
        home_id = g['teams']['home']['team']['id']
        away_name = g['teams']['away']['team']['name']
        home_name = g['teams']['home']['team']['name']
        game_pk = g['gamePk']
        is_day = game_times.get(game_pk, {}).get('is_day', False)

        away_starter = next((s for s in starters if s['teamId'] == away_id), None)
        home_starter = next((s for s in starters if s['teamId'] == home_id), None)
        away_batters = roster_batters.get(away_id, set())
        home_batters = roster_batters.get(home_id, set())
        if not away_starter or not home_starter or not away_batters or not home_batters:
            print(f"Skipping {away_name} @ {home_name} – missing data")
            continue

        # Weather
        stadium_key = next((k for k,v in stadiumData.items() if v['teamId'] == home_id), None)
        stadium = stadiumData.get(stadium_key)
        weather = {'temp':70, 'windSpeed':0, 'windDeg':0}
        if stadium and not stadium['isDome']:
            wx_url = f"https://api.openweathermap.org/data/2.5/weather?lat={stadium['lat']}&lon={stadium['lon']}&appid={OPENWEATHER_API_KEY}&units=imperial"
            wx_data = fetch_json(wx_url)
            if wx_data and 'main' in wx_data:
                weather = {
                    'temp': wx_data['main']['temp'],
                    'windSpeed': wx_data.get('wind',{}).get('speed',0),
                    'windDeg': wx_data.get('wind',{}).get('deg',0),
                }
                print(f"   Weather for {home_name}: {weather['temp']}°F, wind {weather['windSpeed']} mph")
            else:
                print(f"   ⚠️ Weather failed for {home_name}")
        elif stadium and stadium['isDome']:
            print(f"   {home_name} is a dome – weather not needed")
        home_abbrev = stadium['abbrev'] if stadium else home_name[:3].upper()
        park_factor = parkFactorHR.get(home_abbrev, 1.0)

        def build_row(bid, starter):
            b = all_batters.get(bid)
            if not b: return None
            hand = batter_hand.get(bid, '')
            platoon = 1 if hand == 'S' or (hand == 'L' and starter['hand'] == 'R') or (hand == 'R' and starter['hand'] == 'L') else 0
            # Plate discipline
            disc = plate_dict.get(bid, {})
            o_swing = disc.get("o_swing_pct", 0)
            z_swing = disc.get("z_swing_pct", 0)
            contact = disc.get("contact_pct", 0)
            # Platoon splits
            splits = batter_platoon.get(bid, {})
            wOBA_vs_L = splits.get('wOBA_vs_L', 0)
            wOBA_vs_R = splits.get('wOBA_vs_R', 0)
            ISO_vs_L = splits.get('ISO_vs_L', 0)
            ISO_vs_R = splits.get('ISO_vs_R', 0)
            # Batted ball / expected
            hard_hit = expected.get(bid, {}).get('ev95percent', 0)
            gb = b.get('gb', 0)
            fb_ld = b.get('fbld', 0)
            # Pitcher stats
            pitcher_stats = all_pitchers.get(starter['id'], {})
            pitcher_exp_stats = pitcher_exp.get(starter['id'], {})
            pitcher_xwoba = pitcher_exp_stats.get('xwoba', 0)
            pitcher_xslg = pitcher_exp_stats.get('xslg', 0)
            pitcher_xba = pitcher_exp_stats.get('xba', 0)
            pitcher_hr_fb = pitcher_exp_stats.get('hr_fb_rate', 0) or pitcher_stats.get('hr_fb_rate', 0)
            arm = pitcher_arsenal.get(starter['id'], {})
            fb_velo = arm.get('fb_velo', 0)
            spin_rate = arm.get('spin_rate', 0)
            horiz_break = arm.get('horiz_break', 0)
            vert_break = arm.get('vert_break', 0)

            row = {
                'batter_name': b['last_name, first_name'],
                'batter_id': bid,
                'batter_hand': hand,
                'pitcher_name': starter['name'],
                'pitcher_id': starter['id'],
                'pitcher_hand': starter['hand'],
                'home_team': home_abbrev,
                'park_factor_hr': park_factor,
                'temperature': weather['temp'],
                'wind_speed': weather['windSpeed'],
                'wind_direction': weather['windDeg'],
                'is_day_game': 1 if is_day else 0,
                'platoon_advantage': platoon,
                'avg_hit_speed': b.get('avg_hit_speed', 0),
                'barrel_rate': b.get('brl_percent', 0),
                'hot_woba': safe_get(hot_streak.get(bid), 'hot_woba'),
                'hot_iso': safe_get(hot_streak.get(bid), 'hot_iso'),
                'avg_bat_speed': safe_get(batspeed.get(bid), 'avg_bat_speed'),
                'hard_hit_rate': hard_hit,
                'gb_pct': gb,
                'fb_ld_pct': fb_ld,
                'o_swing_pct': o_swing,
                'z_swing_pct': z_swing,
                'contact_pct': contact,
                'wOBA_vs_L': wOBA_vs_L,
                'wOBA_vs_R': wOBA_vs_R,
                'ISO_vs_L': ISO_vs_L,
                'ISO_vs_R': ISO_vs_R,
                'pitcher_xwoba': pitcher_xwoba,
                'pitcher_xslg': pitcher_xslg,
                'pitcher_xba': pitcher_xba,
                'pitcher_hr_fb_rate': pitcher_hr_fb,
                'pitcher_fb_velo': fb_velo,
                'pitcher_spin_rate': spin_rate,
                'pitcher_horiz_break': horiz_break,
                'pitcher_vert_break': vert_break,
                'actual_hr': 0  # no labels for prediction
            }
            return row

        for bid in away_batters:
            row = build_row(bid, home_starter)
            if row: features.append(row)
        for bid in home_batters:
            row = build_row(bid, away_starter)
            if row: features.append(row)

    if not features:
        raise ValueError("No features generated for today.")
    df = pd.DataFrame(features)
    print(f"✅ Built {len(df)} feature rows for {TODAY}.")
    return df

# ----------------------------------------------------------------------------
# LOAD TRAINING DATA FROM GOOGLE SHEET
# ----------------------------------------------------------------------------
def load_training_data():
    print("\n📚 Loading training data from Google Sheet...")
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(SHEET_ID)
    worksheet = sh.sheet1
    data = worksheet.get_all_values()
    if len(data) < 2:
        raise ValueError("Sheet has no training data. Run backfill first.")
    df = pd.DataFrame(data[1:], columns=data[0])
    df['hr_training_date'] = pd.to_datetime(df['hr_training_date'])
    if ROLLING_DAYS:
        cutoff = datetime.now() - timedelta(days=ROLLING_DAYS)
        df = df[df['hr_training_date'] >= cutoff]
    # Convert numeric columns
    numeric_cols = [
        'actual_hr', 'avg_hit_speed', 'barrel_rate', 'hot_woba', 'hot_iso',
        'platoon_advantage', 'park_factor_hr', 'temperature', 'wind_speed',
        'avg_bat_speed', 'pitcher_avg_hit_speed', 'hard_hit_rate', 'gb_pct',
        'fb_ld_pct', 'o_swing_pct', 'z_swing_pct', 'contact_pct',
        'wOBA_vs_L', 'wOBA_vs_R', 'ISO_vs_L', 'ISO_vs_R', 'pitcher_xwoba',
        'pitcher_xslg', 'pitcher_xba', 'pitcher_hr_fb_rate', 'pitcher_fb_velo',
        'pitcher_spin_rate', 'pitcher_horiz_break', 'pitcher_vert_break', 'is_day_game'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df.fillna(0, inplace=True)
    # Feature engineering (power_vs_power is already in training? Not needed, but we can add if desired)
    print(f"Training rows: {len(df)} | Positive: {df['actual_hr'].sum()} | Negative: {len(df)-df['actual_hr'].sum()}")
    return df

# ----------------------------------------------------------------------------
# TRAIN MODEL AND PREDICT
# ----------------------------------------------------------------------------
def train_and_predict(df_train, df_pred):
    # Feature columns (all 34 columns except the target and identifiers)
    feature_cols = [
        'avg_hit_speed', 'barrel_rate', 'hot_woba', 'hot_iso',
        'platoon_advantage', 'park_factor_hr', 'temperature', 'wind_speed',
        'is_day_game', 'avg_bat_speed', 'hard_hit_rate', 'gb_pct', 'fb_ld_pct',
        'o_swing_pct', 'z_swing_pct', 'contact_pct',
        'wOBA_vs_L', 'wOBA_vs_R', 'ISO_vs_L', 'ISO_vs_R',
        'pitcher_xwoba', 'pitcher_xslg', 'pitcher_xba', 'pitcher_hr_fb_rate',
        'pitcher_fb_velo', 'pitcher_spin_rate', 'pitcher_horiz_break', 'pitcher_vert_break'
    ]
    # Ensure all feature columns exist in both dataframes
    for col in feature_cols:
        if col not in df_train.columns:
            df_train[col] = 0
        if col not in df_pred.columns:
            df_pred[col] = 0
    X = df_train[feature_cols]
    y = df_train['actual_hr']

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scale_pos_weight = (len(y_tr) - y_tr.sum()) / y_tr.sum() if y_tr.sum() > 0 else 1

    xgb = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, eval_metric='logloss', use_label_encoder=False)
    param_grid = {'n_estimators': [100, 200], 'max_depth': [4, 6], 'learning_rate': [0.05, 0.1], 'subsample': [0.8, 1.0]}
    grid = GridSearchCV(xgb, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=0)
    grid.fit(X_tr, y_tr)
    best_raw = grid.best_estimator_
    print(f"Best params: {grid.best_params_}, CV AUC: {grid.best_score_:.4f}")

    calibrated = CalibratedClassifierCV(best_raw, method='sigmoid', cv=3)
    calibrated.fit(X_tr, y_tr)
    y_cal = calibrated.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, y_cal)
    print(f"Test AUC: {auc:.4f}")

    # Feature importance
    imp = pd.Series(best_raw.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print("\nTop 10 features:\n", imp.tail(10))

    X_pred = df_pred[feature_cols]
    probs = calibrated.predict_proba(X_pred)[:, 1]
    df_pred['hr_probability'] = probs
    return df_pred

# ----------------------------------------------------------------------------
# DRAFTKINGS ODDS FETCH (WORKING VERSION)
# ----------------------------------------------------------------------------
def fetch_dk_hr_data():
    DK_LEAGUE_ID = "84240"
    DK_HR_SUBCATEGORY_ID = "17319"
    DK_SITE = "US-PA-SB"

    ev_filter = (
        f"$filter=leagueId eq '{DK_LEAGUE_ID}' AND "
        f"clientMetadata/Subcategories/any(s: s/Id eq '{DK_HR_SUBCATEGORY_ID}')"
    )
    mk_filter = (
        f"$filter=clientMetadata/subCategoryId eq '{DK_HR_SUBCATEGORY_ID}' AND "
        f"tags/all(t: t ne 'SportcastBetBuilder')"
    )
    tv_query = f"{DK_LEAGUE_ID},{DK_HR_SUBCATEGORY_ID}"

    ev_q = urllib.parse.quote(ev_filter, safe='')
    mk_q = urllib.parse.quote(mk_filter, safe='')
    tv_q = urllib.parse.quote(tv_query, safe='')

    url = (
        f"https://sportsbook-nash.draftkings.com/sites/{DK_SITE}"
        f"/api/sportscontent/controldata/league/leagueSubcategory/v1/markets"
        f"?isBatchable=false&templateVars={tv_q}"
        f"&eventsQuery={ev_q}"
        f"&marketsQuery={mk_q}"
        f"&include=Events&entity=events"
    )

    headers = {
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Origin": "https://sportsbook.draftkings.com",
        "Referer": (
            "https://sportsbook.draftkings.com/leagues/baseball/mlb"
            "?category=games&subcategory=batter-props&nav_1=home-runs"
        ),
        "User-Agent": (
            "Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like Mac OS X) "
            "AppleWebKit/605.1.15 (KHTML, like Gecko) "
            "Version/18.0 Mobile/15E148 Safari/604.1"
        ),
    }
    r = requests.get(url, headers=headers, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"DraftKings HTTP {r.status_code}: {r.text[:200]}")
    return r.json()

def fetch_draftkings_odds():
    data = fetch_dk_hr_data()
    events     = data.get("events", [])     or []
    markets    = data.get("markets", [])    or []
    selections = data.get("selections", []) or []

    event_map = {}
    for ev in events:
        name = ev.get("name") or f"{ev.get('awayTeamName','')} @ {ev.get('homeTeamName','')}"
        event_map[ev.get("id")] = name

    market_map = {m.get("id"): m for m in markets}

    odds_dict = {}
    seen = set()

    for sel in selections:
        if str(sel.get("label", "")).strip() != "1+":
            continue
        player_name = "Unknown"
        participants = sel.get("participants") or []
        if participants:
            names = [
                (p.get("name") or p.get("displayName") or f"P{p.get('id','')}")
                for p in participants
                if p.get("type") in (None, "Player")
            ]
            if names:
                player_name = ", ".join(names)
        if player_name in seen:
            continue
        seen.add(player_name)
        odds = None
        display_odds = sel.get("displayOdds") or {}
        if isinstance(display_odds, dict) and display_odds.get("american") is not None:
            odds = display_odds["american"]
        elif sel.get("americanOdds") is not None:
            odds = str(sel["americanOdds"])
        if odds is None:
            continue
        if isinstance(odds, str):
            odds = int(odds.replace('+', ''))
        decimal = (odds / 100) + 1 if odds > 0 else (100 / abs(odds)) + 1
        clean = re.sub(r'[^\w\s]', '', player_name).lower()
        odds_dict[clean] = {"american": odds, "decimal": decimal, "original": player_name}
    print(f"✅ Fetched odds for {len(odds_dict)} players")
    return odds_dict

def add_odds_and_ev(df_pred):
    odds = fetch_draftkings_odds()
    if not odds:
        print("No odds fetched. Skipping EV calculation.")
        df_pred['ev'] = -1
        return df_pred

    def norm_name(name):
        if ',' in name:
            parts = name.split(',')
            name = (parts[1].strip() + ' ' + parts[0].strip()).lower()
        else:
            name = name.lower()
        name = re.sub(r'\s*(jr\.|sr\.|ii|iii|iv)$', '', name)
        name = re.sub(r'[^\w\s]', '', name)
        return name.strip()

    df_pred['norm_name'] = df_pred['batter_name'].apply(norm_name)
    df_pred['decimal_odds'] = 0.0
    df_pred['american_odds'] = 0
    df_pred['ev'] = -1.0

    odds_keys = list(odds.keys())
    from difflib import get_close_matches
    for idx, row in df_pred.iterrows():
        key = row['norm_name']
        if key in odds:
            df_pred.at[idx, 'decimal_odds'] = odds[key]['decimal']
            df_pred.at[idx, 'american_odds'] = odds[key]['american']
            df_pred.at[idx, 'ev'] = row['hr_probability'] * odds[key]['decimal'] - 1
        else:
            match = get_close_matches(key, odds_keys, n=1, cutoff=0.8)
            if match:
                best = match[0]
                df_pred.at[idx, 'decimal_odds'] = odds[best]['decimal']
                df_pred.at[idx, 'american_odds'] = odds[best]['american']
                df_pred.at[idx, 'ev'] = row['hr_probability'] * odds[best]['decimal'] - 1
    return df_pred

# ----------------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------------
def main():
    drive.mount('/content/drive')
    output_folder = "/content/drive/MyDrive/mlb_hr_data/"
    !mkdir -p "$output_folder"

    # 1. Load static data (once)
    plate_dict = fetch_plate_discipline()
    static_data = load_static_csvs()

    # 2. Build today's features
    df_pred = fetch_todays_features(plate_dict, static_data)

    # 3. Load training data from sheet
    df_train = load_training_data()

    # 4. Train and predict
    df_pred = train_and_predict(df_train, df_pred)

    # 5. Add odds and EV
    df_pred = add_odds_and_ev(df_pred)

    # 6. Output results
    print("\n" + "="*80)
    print(f"🔮 TOP 10 MOST LIKELY HOME RUNS FOR {TODAY}")
    print("="*80)
    top10 = df_pred.nlargest(10, 'hr_probability')[['batter_name', 'pitcher_name', 'home_team', 'hr_probability']]
    for _, row in top10.iterrows():
        print(f"{row['batter_name']:25} vs {row['pitcher_name']:25} ({row['home_team']}) → {row['hr_probability']:.3f} ({row['hr_probability']*100:.1f}%)")

    print("\n" + "="*80)
    print(f"💰 TOP 10 HIGHEST EXPECTED VALUE BETS FOR {TODAY} (EV > 0)")
    print("="*80)
    positive_ev = df_pred[df_pred['ev'] > 0].sort_values('ev', ascending=False)
    if len(positive_ev) == 0:
        print("No +EV bets found today.")
    else:
        for _, row in positive_ev.head(10).iterrows():
            print(f"{row['batter_name']:25} vs {row['pitcher_name']:25} ({row['home_team']})")
            print(f"   Model Prob: {row['hr_probability']:.3f} ({row['hr_probability']*100:.1f}%) | DK Odds: {row['american_odds']:+d} | EV: {row['ev']:.3f}\n")

    # 7. Save full results to Drive
    output_path = f"{output_folder}hr_predictions_with_odds_{TODAY}.csv"
    df_pred.to_csv(output_path, index=False)
    print(f"\n✅ Full results saved to {output_path}")

if __name__ == "__main__":
    main()

Mounted at /content/drive
✅ Fetched plate discipline for 163 batters.
Loading static Statcast CSVs...
Loaded: batters=259, pitchers=368, expected=259, batspeed=213, pitcher_exp=368, arsenal=396

📡 Fetching today's schedule (2026-05-29) and building features...
Found 15 games.
Probable starters: 30
Grant Holmes: R
Chris Paddack: R
Paxton Schultz: R
Lucas Giolito: R
Taj Bradley: R
Jared Jones: R
Adam Macko: L
Tyler Samaniego: L
Slade Cecconi: R
Trevor Rogers: L
Nick Martinez: R
Andre Pallante: R
Troy Melton: R
Erick Fedde: R
Stephen Kolek: R
Shota Imanaga: L
MacKenzie Gore: L
Freddy Peralta: R
Max Meyer: R
Walbert Ureña: R
Coleman Crow: R
Logan Webb: R
Kai-Wei Teng: R
Carlos Rodón: L
George Kirby: R
Michael Lorenzen: R
Luis Severino: R
Zack Wheeler: R
Zac Gallen: R
Justin Wrobleski: L

Fetching active rosters...
Team 134: 13 batters
Team 135: 13 batters
Team 138: 13 batters
Team 142: 13 batters
Team 139: 13 batters
Team 133: 13 batters
Team 140: 13 batters
Team 137: 12 batters
Team 136: 